In [6]:
import pandas as pd
books = pd.read_csv("books_cleaned.csv")

In [7]:
category_mapping = {
    'Fiction': "fictional story or novel",
    'Juvenile Fiction': "Children's fiction",
    'Biography': "non-fiction or informational book",
    'Literary Criticism': "non-fiction or informational book",
    'History': "non-fiction or informational book",
    'Philosophy': "non-fiction or informational book",
    'Religion': "non-fiction or informational book",
    'Comics & Graphic Novels': "fictional story or novel",
    'Drama': "fictional story or novel",
    'Juvenile Nonfiction': "Children's non_fiction",
    'Science': "non-fiction or informational book",
    'Poetry': "fictional story or novel"
}
books["simple_categories"] = books["categories"].map(category_mapping)


In [8]:
import torch
from transformers import pipeline

# Step 1: Check if CUDA is actually available to PyTorch
# device 0 = First GPU, device -1 = CPU
device_id = 0 if torch.cuda.is_available() else -1

if device_id == 0:
    print("Nice! Using GPU (CUDA).")
else:
    print("CUDA not found by PyTorch. Falling back to CPU.")

# Step 2: Initialize the pipeline
fiction_categories = ["fictional story or novel",
        "non-fiction or informational book"]

pipe = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device_id  # Use the integer ID here
)

C:\Users\chake\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA not found by PyTorch. Falling back to CPU.


Device set to use cpu


In [9]:
sequence = books.loc[books["simple_categories"] == "fictional story or novel", "description"].reset_index(drop=True)[0]

In [10]:
pipe(sequence, fiction_categories)

{'sequence': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst

In [11]:
import pandas as pd 
import numpy as np 

result = pipe(sequence, fiction_categories)
max_index = np.argmax(result["scores"])
max_label = result["labels"][max_index]
max_label

'fictional story or novel'

In [12]:
def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label

In [17]:
from transformers import pipeline
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

fiction_books = books[books["simple_categories"] == "fictional story or novel"].head(150)
nonfiction_books = books[books["simple_categories"] == "non-fiction or informational book"].head(150)
test_books = pd.concat([fiction_books, nonfiction_books]).sample(frac=1, random_state=42).reset_index(drop=True)

descriptions = test_books["description"].tolist()
actual_cats = test_books["simple_categories"].tolist()

# OPTIMIZED: Use larger batch size for speed
predicted_cats = []
batch_size = 16  # Increased from 16 (test if your GPU can handle it)

print("Classifying books...")
for i in tqdm(range(0, len(descriptions), batch_size)):
    batch = descriptions[i:i+batch_size]
    
    results = pipe(
        batch,
        candidate_labels=["fictional story or novel", "non-fiction or informational book"],
        hypothesis_template="This book is a {}.",
        multi_label=False
    )
    
    predicted_cats.extend([r['labels'][0] for r in results])

# Calculate accuracy
accuracy = accuracy_score(actual_cats, predicted_cats)
print(f"\n🎯 Accuracy: {accuracy:.3f}")
print("\n" + classification_report(actual_cats, predicted_cats))

Classifying books...


100%|██████████| 19/19 [11:24<00:00, 36.03s/it]



🎯 Accuracy: 0.860

                                   precision    recall  f1-score   support

         fictional story or novel       0.94      0.77      0.85       150
non-fiction or informational book       0.81      0.95      0.87       150

                         accuracy                           0.86       300
                        macro avg       0.87      0.86      0.86       300
                     weighted avg       0.87      0.86      0.86       300



In [18]:
from tqdm import tqdm
import pandas as pd

# Get books with missing categories
missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

print(f"Found {len(missing_cats)} books with missing categories")

# Extract data once
isbns = missing_cats["isbn13"].tolist()
descriptions = missing_cats["description"].tolist()

# Batch prediction (MUCH FASTER)
predicted_cats = []
batch_size = 32  # Process 32 books at once

print("Classifying books with missing categories...")
for i in tqdm(range(0, len(descriptions), batch_size)):
    batch_descriptions = descriptions[i:i+batch_size]
    
    # Process entire batch at once
    results = pipe(
        batch_descriptions,
        candidate_labels=["fictional story or novel", "non-fiction or informational book"],
        hypothesis_template="This book is a {}.",
        multi_label=False
    )
    
    # Extract predictions
    predicted_cats.extend([r['labels'][0] for r in results])

# Create results dataframe
results_df = pd.DataFrame({
    "isbn13": isbns,
    "predicted_category": predicted_cats
})

print(f"\n✅ Classified {len(results_df)} books")
print("\nCategory distribution:")
print(results_df["predicted_category"].value_counts())

# Update original dataframe
books.loc[books["simple_categories"].isna(), "simple_categories"] = results_df["predicted_category"].values

# Save updated books
books.to_csv("books_with_predictions.csv", index=False)
print("\n💾 Saved updated books to 'books_with_predictions.csv'")

Found 1765 books with missing categories
Classifying books with missing categories...


100%|██████████| 56/56 [54:53<00:00, 58.81s/it]



✅ Classified 1765 books

Category distribution:
predicted_category
non-fiction or informational book    1318
fictional story or novel              447
Name: count, dtype: int64

💾 Saved updated books to 'books_with_predictions.csv'
